# Skill Evaluation Tutorial

This tutorial walks through evaluating an agent skill using `benchflow skills eval`.

We use the **benchmark-hallucination-audit** skill as a real-world example — a skill that teaches agents to verify claims in benchmark comparison tables by checking papers, GitHub, and HuggingFace.

## Install

```bash
uv tool install benchflow
```

## 1. Skill Structure

A skill with evals looks like this:

```
benchmark-hallucination-audit/
├── skill.md              # Skill instructions
└── evals/
    └── evals.json        # Test cases with ground truth
```

The `skill.md` contains the methodology. The `evals/evals.json` contains test cases that verify the skill actually helps agents.

## 2. The Skill: `skill.md`

This skill teaches a 5-round layered subagent audit process:

| Round | What | Agents | Time |
|-------|------|--------|------|
| 0 | Extract table from paper | 1 | ~10 min |
| 1 | Verify cited papers exist | 4 parallel | ~20 min |
| 2 | Abstract-level cell scan | 4 parallel | ~30 min |
| 3 | Deep 4-source audit (paper+GitHub+HF+website) | 6×5 rows | ~60 min/batch |
| 4 | Correction round (overturn false positives) | 1 | ~15 min |
| 5 | Self-audit (verify paper's own claims) | 1 | ~15 min |

**Key insight**: Abstract-only passes produce ~30-50% false positives. The deep 4-source audit corrects them.

### Common error patterns the skill detects:

1. **Cross-Domain inflation** — task types (OS/DB/Web) ≠ professional domains (medicine/law/finance)
2. **Dynamic aspiration vs mechanism** — "we plan to update" ≠ live leaderboard
3. **Expert Val scope** — annotators ≠ domain experts co-developing criteria
4. **Multi-Modal confusion** — agent interface modality ≠ data input modality
5. **Production generosity** — "inspired by real workflows" ≠ real commercial deployments
6. **Diverse Eval conflation** — metrics (accuracy+F1) ≠ paradigms (execution+rubric+LLM-judge)
7. **Self-grading bias** — paper's own row always gets all ✓

## 3. The Eval Cases: `evals/evals.json`

Each test case has verified ground truth from a real audit of AlphaEval (arXiv:2604.12162) — a paper with a 94-row benchmark comparison table.

In [ ]:
import json
from pathlib import Path

# Load and display the eval cases
evals = json.loads(Path("benchmark-hallucination-audit/evals/evals.json").read_text())
print(f"Skill: {evals['skill_name']}")
print(f"Cases: {len(evals['cases'])}")
print(f"Judge model: {evals['defaults']['judge_model']}")
print(f"Timeout: {evals['defaults']['timeout_sec']}s")
print()
for case in evals['cases']:
    verdict = 'OVERCLAIM' if 'overclaim' in case['id'] else ('MISSING' if 'missing' in case['id'] else 'VERIFY')
    print(f"  {case['id']:40s}  {verdict}")

Expected output:
```
Skill: benchmark-hallucination-audit
Cases: 8
Judge model: claude-haiku-4-5-20251001
Timeout: 600s

  overclaim-xdom-agentbench                OVERCLAIM
  overclaim-xdom-videomme                  OVERCLAIM
  missing-dyn-gaia2                        MISSING
  missing-mm-mlebench                      MISSING
  correct-prod-swelancer                   VERIFY
  overclaim-prod-xbench                    OVERCLAIM
  missing-exp-swelancer                    MISSING
  self-audit-alphaeval-dynamic             VERIFY
```

### Anatomy of a test case

Each case has 4 fields:

In [ ]:
case = evals['cases'][0]  # overclaim-xdom-agentbench
print(f"ID: {case['id']}")
print(f"\nQuestion (sent to agent):")
print(f"  {case['question'][:120]}...")
print(f"\nGround truth (used by LLM judge):")
print(f"  {case['ground_truth'][:120]}...")
print(f"\nExpected behavior (rubric items):")
for b in case['expected_behavior']:
    print(f"  - {b}")

Expected output:
```
ID: overclaim-xdom-agentbench

Question (sent to agent):
  AlphaEval (arXiv:2604.12162) Table 1 marks AgentBench (Liu et al., 2023, arXiv:2308.03688) with Cross-Domain=✓. The...

Ground truth (used by LLM judge):
  OVERCLAIM. AgentBench's 8 environments are TASK TYPES (OS interaction, database queries, knowledge graph reasoning, c...

Expected behavior (rubric items):
  - The agent fetched the AgentBench paper (arXiv:2308.03688) to verify the 8 environments
  - The agent compared the environments against the strict Cross-Domain definition (3+ professional domains)
  - The agent correctly identified that task types ≠ professional domains
  - The agent concluded this is an overclaim
```

## 4. Run the eval

```bash
$ benchflow skills eval ./benchmark-hallucination-audit/ \
    -a claude-agent-acp \
    -a codex-acp \
    -e docker
```

Expected output:
```
Skill eval: benchmark-hallucination-audit (8 cases)
  Agents: claude-agent-acp, codex-acp
  Environment: docker

     Skill Eval: benchmark-hallucination-audit
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┓
┃ Agent             ┃ Mode       ┃ Score ┃ Avg Reward ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━┩
│ claude-agent-acp  │ with-skill │ 7/8   │ 0.88       │
│ claude-agent-acp  │ baseline   │ 3/8   │ 0.42       │
│ claude-agent-acp  │ LIFT       │ +4    │ +0.46      │
│ codex-acp         │ with-skill │ 5/8   │ 0.71       │
│ codex-acp         │ baseline   │ 2/8   │ 0.35       │
│ codex-acp         │ LIFT       │ +3    │ +0.36      │
└───────────────────┴────────────┴───────┴────────────┘
```

The lift shows how much the skill improves agent performance. Without the skill, agents miss many overclaims and incorrectly accept table values at face value.

## 5. What happens under the hood

```
benchflow skills eval ./my-skill/ -a claude-agent-acp
        │
        ▼
┌─ Load evals/evals.json (8 cases)
│
├─ Generate ephemeral tasks (with-skill)
│   ├── overclaim-xdom-agentbench/
│   │   ├── instruction.md      ← case.question
│   │   ├── task.toml           ← timeout from defaults
│   │   ├── environment/
│   │   │   ├── Dockerfile      ← python:3.12 + skill installed
│   │   │   └── skills/         ← SKILL.md + scripts copied in
│   │   └── tests/
│   │       ├── case.json       ← ground_truth + rubric
│   │       ├── judge.py        ← LLM judge template
│   │       └── test.sh         ← entry point
│   └── ... (8 tasks total)
│
├─ Generate ephemeral tasks (baseline — no skill)
│   └── ... (8 tasks, no skills/ directory)
│
├─ Run Job engine (with-skill)
│   ├── Start Docker container per task
│   ├── Install agent via ACP
│   ├── Send instruction.md as prompt
│   ├── Agent works (reads SKILL.md, fetches papers, reasons)
│   ├── Verifier runs judge.py
│   │   └── LLM grades trajectory against rubric → reward 0.0-1.0
│   └── Collect result.json per task
│
├─ Run Job engine (baseline)
│   └── Same but without skill installed
│
├─ Compute lift per agent
│   └── with_skill_score - baseline_score
│
├─ Display results table
│
└─ Cleanup ephemeral tasks (auto-deleted)
```

## 6. Export for skill improvement (GEPA)

```bash
$ benchflow skills eval ./benchmark-hallucination-audit/ \
    -a claude-agent-acp \
    --export-gepa
```

This creates traces for the SkillSpin improvement pipeline:

```
jobs/skill-eval/benchmark-hallucination-audit/gepa/
├── skill.md                                          # Current skill snapshot
├── traces/
│   ├── overclaim-xdom-agentbench-claude-agent-acp-with.json
│   ├── overclaim-xdom-agentbench-claude-agent-acp-without.json
│   ├── missing-dyn-gaia2-claude-agent-acp-with.json
│   └── ...
└── summary.json                                      # Aggregate lift scores
```

Each trace file contains the agent's trajectory, score, rubric results, and the skill text — everything needed to evolve the skill automatically.

## 7. Programmatic API

You can also run skill eval from Python:

In [ ]:
import asyncio
from benchflow.skill_eval import SkillEvaluator, export_gepa_traces

# Load and validate
evaluator = SkillEvaluator("./benchmark-hallucination-audit/")
print(f"Skill: {evaluator.dataset.skill_name}")
print(f"Cases: {len(evaluator.dataset.cases)}")

# Run (requires Docker + API keys)
# result = asyncio.run(evaluator.run(
#     agents=["claude-agent-acp", "codex-acp"],
#     environment="docker",
#     concurrency=2,
# ))
#
# # Display results
# for row in result.summary_table():
#     print(f"{row['agent']:20s} {row['mode']:12s} {row['score']:6s} {row['avg_reward']}")
#
# # Export for improvement
# export_gepa_traces(result, evaluator.dataset, "traces/")

## 8. Writing your own eval cases

```json
{
  "skill_name": "your-skill",
  "defaults": {
    "timeout_sec": 300,
    "judge_model": "claude-haiku-4-5-20251001"
  },
  "cases": [
    {
      "id": "case-001",
      "question": "Clear task for the agent",
      "ground_truth": "Expected correct answer",
      "expected_behavior": [
        "Agent did step 1",
        "Agent did step 2",
        "Agent produced correct output"
      ]
    }
  ]
}
```

**Tips:**
- 2-8 cases is the sweet spot
- Each rubric item should be independently verifiable from the trajectory
- Include cases where the correct answer is "no action needed" (baseline shouldn't score 100%)
- Test the lift, not just correctness — if baseline already scores high, the skill isn't helping